# Industry Job Advertisement Analysis

Compare quarterly job-advertisement indices and annual growth across
ten industries. Indices measure relative changes, not vacancy counts
or industry market shares.

In [1]:
from pathlib import Path

import pandas as pd

industry = pd.read_csv(
    Path("../data/processed/industry_quarterly.csv"),
    parse_dates=["Date"]
)

industry.head()

,Date,Accounting,Construction,Education,Health,Hospitality,IT,Manufacturing,Primary,Sales,Other
0,2010-12-01,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
1,2011-03-01,124.6,122.5,56.8,113.6,123.8,114.4,121.0,130.6,125.1,123.5
2,2011-06-01,137.6,134.3,55.0,124.6,108.2,126.1,122.4,131.6,128.8,124.2
3,2011-09-01,138.6,144.6,64.6,129.0,131.3,125.7,140.7,156.6,134.4,137.9
4,2011-12-01,107.2,122.3,84.6,107.0,103.0,106.0,116.0,143.1,104.9,121.1


In [2]:
industry_indices = industry.set_index("Date")

industry_yoy = industry_indices.pct_change(
    periods=4,
    fill_method=None
) * 100

industry_yoy.tail()

,Accounting,Construction,Education,Health,Hospitality,IT,Manufacturing,Primary,Sales,Other
Date,,,,,,,,,,
2025-06-01,-13.394755,-7.206804,-14.223003,13.051146,-8.806996,-5.000000,-19.076392,2.316749,-14.862543,-1.435726
2025-09-01,0.777605,10.023419,-13.156101,4.242156,6.763285,10.192308,-0.230734,2.871972,-0.802855,12.383178
2025-12-01,2.865613,10.549451,-13.570939,15.675676,-0.283286,13.429257,2.863092,16.737189,10.095012,9.133307
2026-03-01,6.098514,21.974371,-6.706114,18.839885,1.995906,12.474849,11.419887,21.406610,12.121212,5.393676
2026-06-01,1.800327,13.941148,-6.755374,2.691108,-8.561644,7.602339,19.146667,9.094284,8.173562,9.044715


In [3]:
latest_industry_growth = industry_yoy.iloc[-1].sort_values(
    ascending=False
)

print("Quarter:", industry_yoy.index[-1].to_period("Q"))
latest_industry_growth.round(2)

Quarter: 2026Q2


Manufacturing    19.15
Construction     13.94
Primary           9.09
Other             9.04
Sales             8.17
IT                7.60
Health            2.69
Accounting        1.80
Education        -6.76
Hospitality      -8.56
Name: 2026-06-01 00:00:00, dtype: float64

### Compare index levels with annual growth

In [4]:
industry_summary = pd.DataFrame({
    "index_value": industry_indices.iloc[-1],
    "yoy_change_pct": industry_yoy.iloc[-1]
})

industry_summary = industry_summary.sort_values(
    "yoy_change_pct",
    ascending=False
)

industry_summary.round(2)

,index_value,yoy_change_pct
Manufacturing,223.4,19.15
Construction,236.2,13.94
Primary,293.9,9.09
Other,321.9,9.04
Sales,107.2,8.17
IT,55.2,7.60
Health,263.3,2.69
Accounting,124.4,1.80
Education,91.1,-6.76
Hospitality,133.5,-8.56


In [5]:
industry_filtered = industry_summary.loc[
    (industry_summary["yoy_change_pct"] > 0) 
    & (industry_summary["index_value"] < 100)
]

In [6]:
industry_filtered

,index_value,yoy_change_pct
IT,55.2,7.602339


In [7]:
it_comparison = pd.DataFrame({
    "index_value": industry_indices["IT"],
    "yoy_change_pct": industry_yoy["IT"],
    "qoq_change_pct": (
        industry_indices["IT"].pct_change(
            periods=1,
            fill_method=None
        ) * 100
    )
})

it_comparison.tail(5).round(2)

,index_value,yoy_change_pct,qoq_change_pct
Date,,,
2025-06-01,51.3,-5.00,3.22
2025-09-01,57.3,10.19,11.70
2025-12-01,47.3,13.43,-17.45
2026-03-01,55.9,12.47,18.18
2026-06-01,55.2,7.60,-1.25


IT's job-advertisement index increased by 7.60% year over year in
2026Q2 but declined by 1.25% quarter over quarter. The index was
higher than in 2025Q2 despite falling from 2026Q1. Quarterly changes
should be interpreted cautiously because the series is unadjusted.

In [8]:
it_comparison.loc[
    (it_comparison["yoy_change_pct"] > 0) 
    & (it_comparison["qoq_change_pct"] < 0)
].tail(5).round(2)

,index_value,yoy_change_pct,qoq_change_pct
Date,,,
2018-12-01,92.8,14.99,-16.09
2019-12-01,97.7,5.28,-15.85
2021-12-01,121.0,29.69,-23.22
2025-12-01,47.3,13.43,-17.45
2026-06-01,55.2,7.60,-1.25


### Conclusion: Industry growth and comparison periods

In 2026Q2, eight of ten industries recorded positive annual growth.
Manufacturing led at 19.15%, while Hospitality recorded the largest
decline at 8.56%.

IT ranked sixth for annual growth at 7.60%, but its index remained
below its December 2010 baseline at 55.2. IT also declined by 1.25%
compared with the preceding quarter.

Positive annual growth can coexist with negative quarterly growth
because the measures use different comparison periods. These
unadjusted indices describe relative changes in advertisements,
not vacancy counts or industry market shares.